# Baseline (SED, EfficientNet-B0)


In [1]:
import os, ast, math, random, time, gc, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torchaudio, timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

## Config

In [11]:
class CFG:
    seed = 42
    sr = 32000
    duration = 15
    n_samples = sr * duration
    window_sec = 5
    window_samples = sr * window_sec

    n_fft = 2048; hop_length = 512; n_mels = 128; fmin = 20; fmax = 16000
    model_name = "tf_efficientnet_b0.ns_jft_in1k"
    in_channels = 1; pretrained = True
    epochs = 3; batch_size = 32; lr = 1e-3; weight_decay = 1e-5
    warmup_epochs = 2; n_folds = 3

    mixup_alpha = 0.5; mixup_prob = 0.6
    freq_mask = 20; time_mask = 80; noise = 0.005

    clip_loss_weight = 1.0; frame_loss_weight = 0.5

    use_secondary = True; sec_weight = 0.5; label_smooth = 0.01
    min_rating = 2.0; use_soundscapes = True; num_workers = 2

    resume = False
    resume_lr = 3e-4

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); torch.backends.cudnn.benchmark = True

## Dataset

In [3]:
class BirdDS(Dataset):
    def __init__(self, df, sp2i, cfg, mode="train"):
        self.df = df.reset_index(drop=True)
        self.sp2i = sp2i; self.nc = len(sp2i); self.cfg = cfg; self.mode = mode

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        ss = r.get("start_sec", None)
        if isinstance(ss, float) and np.isnan(ss): ss = None
        w = self._load(r["filepath"], ss)
        if self.mode == "train":
            if random.random() < .5: w = w + torch.randn_like(w) * self.cfg.noise
            if random.random() < .5: w = w * random.uniform(0.7, 1.3)
        return w, self._lbl(r)

    def _load(self, fp, ss):
        try:
            if ss is not None:
                w, sr = torchaudio.load(fp, frame_offset=int(float(ss) * self.cfg.sr),
                                        num_frames=self.cfg.n_samples)
            else:
                w, sr = torchaudio.load(fp)
            if w.shape[0] > 1: w = w.mean(0, keepdim=True)
            w = w.squeeze(0)
            if sr != self.cfg.sr: w = torchaudio.functional.resample(w, sr, self.cfg.sr)
        except Exception:
            w = torch.zeros(self.cfg.n_samples)

        if w.shape[0] > self.cfg.n_samples:
            s = (random.randint(0, w.shape[0] - self.cfg.n_samples) if self.mode == "train"
                 else (w.shape[0] - self.cfg.n_samples) // 2)
            w = w[s:s + self.cfg.n_samples]
        elif w.shape[0] < self.cfg.n_samples:
            w = F.pad(w, (0, self.cfg.n_samples - w.shape[0]))
        return w

    def _lbl(self, r):
        y = np.zeros(self.nc, np.float32)
        if isinstance(r["primary_label"], str):
            for p in r["primary_label"].split(";"):
                p = p.strip()
                if p in self.sp2i: y[self.sp2i[p]] = 1.0
        if self.cfg.use_secondary and "secondary_labels" in r.index:
            s = r["secondary_labels"]
            if isinstance(s, str) and s not in ("", "[]"):
                try: sl = ast.literal_eval(s)
                except Exception: sl = [x.strip() for x in s.split(";") if x.strip()]
                for x in sl:
                    if x in self.sp2i: y[self.sp2i[x]] = self.cfg.sec_weight
        return torch.from_numpy(y)

## Mel + Aug

In [4]:
class GPUMel(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            c.sr, c.n_fft, hop_length=c.hop_length, n_mels=c.n_mels,
            f_min=c.fmin, f_max=c.fmax, power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        m = self.db(self.mel(x))
        m = (m - m.mean((-2, -1), keepdim=True)) / (m.std((-2, -1), keepdim=True) + 1e-6)
        return m.unsqueeze(1)

def spec_aug(m, fm=20, tm=80):
    B, C, Fq, T = m.shape
    for i in range(B):
        if random.random() < .5:
            f = random.randint(0, fm); f0 = random.randint(0, max(0, Fq - f)); m[i, :, f0:f0 + f, :] = 0
        if random.random() < .5:
            t = random.randint(0, tm); t0 = random.randint(0, max(0, T - t)); m[i, :, :, t0:t0 + t] = 0
    return m

def wave_mixup(x, y, alpha=0.5):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

## Model

In [5]:
class LSEPool(nn.Module):
    def __init__(self, r=5.0):
        super().__init__()
        self.r = nn.Parameter(torch.tensor(r))
    def forward(self, x):
        return torch.logsumexp(self.r * x, dim=1) / self.r

class SEDModel(nn.Module):
    def __init__(self, cfg, nc):
        super().__init__()
        self.backbone = timm.create_model(cfg.model_name, pretrained=cfg.pretrained,
                                           in_chans=cfg.in_channels, num_classes=0, global_pool="")
        with torch.no_grad():
            o = self.backbone(torch.randn(1, cfg.in_channels, cfg.n_mels, 938))
            if o.dim() == 4: _, c, h, _ = o.shape; self.fd = c * h
            else: self.fd = o.shape[-1]
        self.fc = nn.Linear(self.fd, nc)
        self.lse = LSEPool(r=5.0)
        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            b, c, h, w = f.shape
            f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)
        f = self.drop(f)
        framewise = self.fc(f)
        clipwise = self.lse(framewise)
        framewise_max = framewise.max(dim=1).values
        return clipwise, framewise_max

## Лосс і метрика

In [6]:
class SEDLoss(nn.Module):
    def __init__(self, clip_w=1.0, frame_w=0.5, ls=0.01):
        super().__init__()
        self.clip_w = clip_w; self.frame_w = frame_w; self.ls = ls
    def forward(self, clipwise, framewise_max, target):
        if self.ls > 0:
            target = target * (1 - self.ls) + self.ls / 2
        loss_clip = F.binary_cross_entropy_with_logits(clipwise, target)
        loss_frame = F.binary_cross_entropy_with_logits(framewise_max, target)
        return self.clip_w * loss_clip + self.frame_w * loss_frame

def pcmap(yt, yp, pad=5):
    nc = yt.shape[1]; aps = []
    for i in range(nc):
        t = np.concatenate([yt[:, i], np.ones(pad)])
        p = np.concatenate([yp[:, i], np.ones(pad)])
        try: aps.append(average_precision_score(t, p))
        except Exception: aps.append(0.)
    return float(np.mean(aps))

## Трен

In [7]:
def cosine_sched(opt, wu, total):
    def f(s):
        if s < wu: return s / max(1, wu)
        p = (s - wu) / max(1, total - wu)
        return max(0, .5 * (1 + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train_ep(model, mel_fn, dl, opt, sched, crit, dev, cfg):
    model.train(); sc = GradScaler(); tl = 0.
    for w, tg in tqdm(dl, leave=False, desc="train"):
        w, tg = w.to(dev), tg.to(dev)
        if random.random() < cfg.mixup_prob:
            w, tg = wave_mixup(w, tg, cfg.mixup_alpha)
        with torch.no_grad(): m = mel_fn(w)
        m = spec_aug(m, cfg.freq_mask, cfg.time_mask)
        opt.zero_grad()
        with autocast():
            clip, frame_max = model(m)
            loss = crit(clip, frame_max, tg)
        sc.scale(loss).backward(); sc.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sc.step(opt); sc.update(); sched.step(); tl += loss.item()
    return tl / len(dl)

@torch.no_grad()
def val_ep(model, mel_fn, dl, crit, dev):
    model.eval(); ps, ts, tl = [], [], 0.
    for w, tg in dl:
        w, tg = w.to(dev), tg.to(dev)
        m = mel_fn(w)
        with autocast():
            clip, frame_max = model(m)
            tl += crit(clip, frame_max, tg).item()
        logits = 0.5 * clip + 0.5 * frame_max
        ps.append(torch.sigmoid(logits).cpu().numpy()); ts.append(tg.cpu().numpy())
    ps = np.concatenate(ps); ts = (np.concatenate(ts) > .5).astype(int)
    return tl / len(dl), pcmap(ts, ps)

In [8]:
def prep(data_dir, cfg):
    tax = os.path.join(data_dir, "taxonomy.csv")
    sp = sorted(pd.read_csv(tax)["primary_label"].unique().tolist()) if os.path.exists(tax) else None

    df = pd.read_csv(os.path.join(data_dir, "train.csv"))
    adir = os.path.join(data_dir, "train_audio")
    df["filepath"] = df["filename"].apply(lambda x: os.path.join(adir, x))
    df["start_sec"] = np.nan; df["source"] = "audio"
    if cfg.min_rating > 0 and "rating" in df.columns:
        df = df[(df["rating"] >= cfg.min_rating) | (df["rating"] == 0)].reset_index(drop=True)
    if sp is None: sp = sorted(df["primary_label"].unique().tolist())

    tsl = os.path.join(data_dir, "train_soundscapes_labels.csv")
    tsd = os.path.join(data_dir, "train_soundscapes")
    if cfg.use_soundscapes and os.path.exists(tsl) and os.path.isdir(tsd):
        ts = pd.read_csv(tsl); ts["filepath"] = ts["filename"].apply(lambda x: os.path.join(tsd, x))
        def _ps(v):
            if isinstance(v, (int, float)): return float(v)
            s = str(v).strip(); pts = s.split(":")
            try:
                if len(pts) == 3: return int(pts[0]) * 3600 + int(pts[1]) * 60 + float(pts[2])
                if len(pts) == 2: return int(pts[0]) * 60 + float(pts[1])
                return float(s)
            except Exception: return 0.
        ts["start_sec"] = ts["start"].apply(_ps); ts["source"] = "sc"
        if "secondary_labels" not in ts.columns: ts["secondary_labels"] = ""
        cols = ["filepath", "primary_label", "secondary_labels", "start_sec", "source"]
        if "secondary_labels" not in df.columns: df["secondary_labels"] = ""
        df = pd.concat([df[cols], ts[cols]], ignore_index=True)
    elif "secondary_labels" not in df.columns:
        df["secondary_labels"] = ""

    print(f"{len(sp)} classes | {len(df)} samples")
    return df, sp

In [9]:
def train(cfg=None,
          data_dir="/kaggle/input/competitions/birdclef-2026",
          output_dir="/kaggle/working",
          folds=(0, 1, 2)):
    cfg = cfg or CFG()
    set_seed(cfg.seed); os.makedirs(output_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {dev} | Model: {cfg.model_name} | Input: {cfg.duration}s")

    df, sp = prep(data_dir, cfg); nc = len(sp)
    sp2i = {s: i for i, s in enumerate(sp)}
    with open(os.path.join(output_dir, "species_list.txt"), "w") as f:
        f.write("\n".join(sp))

    mel_fn = GPUMel(cfg).to(dev)

    df["_s"] = df["primary_label"].apply(lambda x: x.split(";")[0].strip() if isinstance(x, str) else "unk")
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)
    df["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df, df["_s"])): df.loc[vi, "fold"] = fi

    t0 = time.time()
    for fold in folds:
        print(f'\n{"=" * 50}\nFOLD {fold}\n{"=" * 50}')
        tdf = df[df["fold"] != fold].reset_index(drop=True)
        vdf = df[df["fold"] == fold].reset_index(drop=True)
        tdl = DataLoader(BirdDS(tdf, sp2i, cfg, "train"), cfg.batch_size, shuffle=True,
                         num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
        vdl = DataLoader(BirdDS(vdf, sp2i, cfg, "val"), cfg.batch_size * 2,
                         num_workers=cfg.num_workers, pin_memory=True)
        print(f"Train: {len(tdf)} ({len(tdl)} batches) | Val: {len(vdf)}")

        model = SEDModel(cfg, nc).to(dev)

        ckpt_path = os.path.join(output_dir, f"fold_{fold}_best.pth")
        start_ep, best, bep, pat = 0, 0., 0, 0
        if cfg.resume and os.path.exists(ckpt_path):
            ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
            model.load_state_dict(ck["model_state_dict"])
            best = ck.get("score", 0.); start_ep = ck.get("epoch", 0)
            print(f"  Resumed from ep{start_ep}, pcmAP={best:.4f}")

        if cfg.resume and start_ep > 0:
            actual_lr, remaining, warmup_ep = cfg.resume_lr, cfg.epochs - start_ep, 0
        else:
            actual_lr, remaining, warmup_ep = cfg.lr, cfg.epochs, cfg.warmup_epochs
        opt = torch.optim.AdamW(model.parameters(), lr=actual_lr, weight_decay=cfg.weight_decay)
        sched = cosine_sched(opt, len(tdl) * warmup_ep, len(tdl) * remaining)
        crit = SEDLoss(cfg.clip_loss_weight, cfg.frame_loss_weight, cfg.label_smooth)
        print(f"  LR: {actual_lr} | Epochs: {start_ep + 1}->{cfg.epochs}")

        for ep in range(start_ep, cfg.epochs):
            tl = train_ep(model, mel_fn, tdl, opt, sched, crit, dev, cfg)
            vl, vs = val_ep(model, mel_fn, vdl, crit, dev)
            print(f"Ep {ep + 1}/{cfg.epochs} | TL:{tl:.4f} VL:{vl:.4f} pcmAP:{vs:.4f} | {(time.time()-t0)/60:.0f}min")
            if vs > best:
                best, bep, pat = vs, ep + 1, 0
                torch.save({"model_state_dict": model.state_dict(), "score": best, "epoch": bep,
                            "cfg": {"model_name": cfg.model_name, "n_mels": cfg.n_mels, "sr": cfg.sr,
                                    "duration": cfg.duration, "n_fft": cfg.n_fft, "hop_length": cfg.hop_length,
                                    "fmin": cfg.fmin, "fmax": cfg.fmax, "num_classes": nc,
                                    "in_channels": cfg.in_channels, "window_sec": cfg.window_sec}},
                           ckpt_path)
                print(f"  -> Saved ({best:.4f})")
            else:
                pat += 1
                if pat >= 7: print("  Early stop"); break
        print(f"Fold {fold}: ep{bep} pcmAP={best:.4f}")
        del model, opt, sched; gc.collect(); torch.cuda.empty_cache()

    print(f"\nDone! Total: {(time.time() - t0) / 60:.1f}min")

In [13]:
cfg = CFG()

train(cfg, folds=(1, 2))

Device: cuda | Model: tf_efficientnet_b0.ns_jft_in1k | Input: 15s
234 classes | 36738 samples

FOLD 1
Train: 24492 (765 batches) | Val: 12246
  LR: 0.001 | Epochs: 1->3


train:   0%|          | 0/765 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Ep 1/3 | TL:0.2794 VL:0.0811 pcmAP:0.3151 | 10min
  -> Saved (0.3151)


train:   0%|          | 0/765 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Ep 2/3 | TL:0.0759 VL:0.0690 pcmAP:0.6179 | 52min
  -> Saved (0.6179)


train:   0%|          | 0/765 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b79f0dc7060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Ep 3/3 | TL:0.0687 VL:0.0729 pcmAP:0.6898 | 63min
  -> Saved (0.6898)
Fold 2: ep3 pcmAP=0.6898

Done! Total: 62.7min
